In [ ]:
import georinex as gr
import numpy as np
import xarray as xr
from matplotlib import pyplot as plt

### File 1: GPS-Only, RINEX Version 2

We will use `georinex` to read in RINEX v2 observation data. `georinex` returns data in the form of an `xarray.Dataset`, an object which is very useful for handling multidimensional data. 

In [ ]:
file = r"data/smst0810.26d.gz"
data = gr.load(file, use=['G','E'], fast=True)

Running a Jupyter cell without any variable assignment  (i.e. '`=`') will print the value, useful for investigating the contents of `xarray` objects:

If we just call `data`, it will show us the contents of the file.

#### Questions:

* How many satellites are in the file?
* What is the start time?
* What is the end time?
* What observables are present?
* What other information do we know about the receiver?

#### Plotting GNSS Data

We will now try plotting the GNSS data!

Try something like:
```py
plt.plot(data['L1'])
```

You can use the time data in the `xarray.Dataset` object to make a nice plot of the observed pseudoranges.

Try:

```py
# creates two plots
fig, axs = plt.subplots(2)

# helps with spacing plots
fig.tight_layout()

# plot data on the first axis
axs[0].plot(data['time'], data['C1'])

# plot data on the second axis
axs[1].plot(data['time'], data['P2'])

```



We can also do math very easily with the `georinex` output. 

Try:
```py
plt.plot(data['time'], data['P2'] - data['C1'])
```

Question: What does this difference represent? What do you expect to see?

`xarray` accepts most `numpy` functions, and most `scipy` ones too. Some functions are also built-in to `xarray`.

```py
print( np.mean(data['P2']) )

print( data['P2'].mean() )

```


`xarray` also supports lots of powerful tools, like the ability to interpolate. A particularly nice one is the ability to do rolling means, medians, etc. 

Try:
```py
plt.plot(data['time'], (data['P2'] - data['C1']).rolling(time=int(5*60/30), center=True).median())
```

`xarray` can also calculate derivatives easily.

Try: 
```py
plt.plot(data['time'], data['C1'].differentiate(coord='time', datetime_unit='h')/1000)
```

Question: What does this show?

#### File 2: Multi-constellation RINEX v3

We will use `georinex` to read in RINEX v3 observation data. 

In [ ]:
file = r"data/ARUA00UGA_S_20260810000_01D_30S_MO.crx.gz"
data = gr.load(file, fast=True)

Examine the contents of the file.

#### Questions:

* How many satellites are in the file?
* What is the start time?
* What is the end time?
* What observables are present?
* What other information do we know about the receiver?

In [ ]:
data

We can use `xarray` to select subsets of the data:

Questions:

* What do you see?


In [ ]:
# selects only data for satellite G03
temp_data = data.sel(sv='G03')

for obs in temp_data:
    
    # checks if the observable has data
    if not np.all(np.isnan(temp_data[obs])):
        print(f"{obs} is valid")
    else:
        print(f"{temp_data['sv'].data} has no data for {obs}")
        continue
    
    fig, axs = plt.subplots(1)

    axs.plot(temp_data['time'], temp_data[obs])
    axs.set_title(f"{obs}")

We can also select multiple satellites, observations, etc. `xarray` is very powerful!

In [ ]:
# list all GPS satelites in the file
list_gps = [s for s in data.sv.values if 'G' in s]

# select only GPS data
temp_data = data.sel(sv=list_gps)

# find obs without any data
list_nan_obs = [obs for obs in temp_data if np.all(np.isnan(temp_data[obs]))]

# drop obs without any data
temp_data = temp_data.drop_vars(list_nan_obs)

In [ ]:
temp_data

#### Exercise:

Can you plot all the pseudorange observables for the Galileo satellites in the file?

#### File 3: RINEX v2 Navigation Data

We will use `georinex` to read in RINEX v2 navigation data. 

In [ ]:
file = r"data/brdc0810.26n.gz"
nav = gr.load(file, fast=True)

Examine the contents of the file.

#### Questions:

* How many satellites are in the file?
* What is the start time?
* What is the end time?
* What other information is there?

This python file has tools which we can use to interpret the navigation data to make an orbit.

In [ ]:
import gnss_tools

The function `gnss_tools.nav2orbit()` will produce an `xarray.Dataset` with the ECEF X, Y, and Z coordinates of the satellite. By default it will create the orbit data at 30 seconds resolution for the whole day.

Try:
```py
orbit = gnss_tools.nav2orbit(nav, time_interval=np.timedelta64(30, 's'))
```

Explore the orbit data!

We can plot the orbital data using `matplotlib`:

Try: 
```py
ax = plt.figure().add_subplot(projection='3d')
for sv in orbit['sv']:
    orb = orbit.sel(sv=sv)
    ax.plot(orb['X'], orb['Y'], orb['Z'])
```